# Fase 3 · Preparación y modelado de datos
**Proyecto BI: Airbnb Bogotá** (Inside Airbnb, captura 2026-09-16)

**Responsabilidad:** convertir `listings_bogota.csv` en el modelo dimensional y las tablas de trazabilidad.
No calcula KPIs ni pruebas (eso es de las fases 4 a 6).

| | |
|---|---|
| **Entrada** | `datos/listings_bogota.csv` (CSV plano, sin descompresión) |
| **Salida** | carpeta `salida_fase3/` (CSV UTF-8 con BOM + `informe_preparacion.json`) |
| **Siguiente** | `02_kpis_dashboard.ipynb` |

Tablas generadas: `fact_anuncios`, `dim_localidad`, `dim_tipo_alojamiento`, `dim_anfitrion`,
`airbnb_bogota_analitico` (plana), `anfitriones_resumen`, `muestra_h1_localidad`,
`muestra_h2_tipo_alojamiento`, `anuncios_excluidos`, `calidad_variables`, `registro_limpieza`,
`diccionario_datos`.

## 0. Configuración y carga

In [1]:
import hashlib
import json
import re

import numpy as np
import pandas as pd

from bi_airbnb import (ENTRADA, DIR_FASE3, MIN_N, Registro, exigir, filtrar_min_n,
                       guardar_csv)

DIR_FASE3.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
registrar = Registro()   # bitácora de decisiones de limpieza

raw = pd.read_csv(ENTRADA, low_memory=False, dtype={"id": "string", "host_id": "string"})
registrar("carga", "registros y columnas originales", f"{raw.shape[0]} x {raw.shape[1]}")

[carga] registros y columnas originales: 19505 x 90


## 1. Selección de variables

In [2]:
COLUMNAS = [
    "id", "host_id", "host_is_superhost", "host_identity_verified",
    "hosts_time_as_host_years", "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "neighbourhood_cleansed", "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bedrooms", "beds",
    "bathrooms_text", "amenities", "minimum_nights", "availability_365",
    "license", "price", "number_of_reviews", "number_of_reviews_ltm",
    "number_of_reviews_l30d", "reviews_per_month", "first_review", "last_review",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "review_scores_value",
    "estimated_occupancy_l365d", "estimated_revenue_l365d", "last_scraped",
]
vacias = [c for c in raw.columns if raw[c].isna().all()]
registrar("seleccion", "columnas 100 % vacías descartadas", len(vacias))
faltantes = sorted(set(COLUMNAS) - set(raw.columns))
exigir(not faltantes, f"Faltan columnas necesarias en el archivo detallado: {faltantes}")
df = raw[COLUMNAS].copy()
registrar("seleccion", f"variables conservadas de las {raw.shape[1]} originales", len(COLUMNAS))

[seleccion] columnas 100 % vacías descartadas: 14
[seleccion] variables conservadas de las 90 originales: 35


## 2. Calidad: identificadores, precio y valores atípicos
El precio se interpreta con el formato de la fuente (coma de miles, punto decimal) y se conserva el texto original.
Los atípicos se detectan con 1,5 × RIQ sobre `log(precio)`, criterio de la Fase 2.

In [3]:
registrar("calidad", "id duplicados", int(df["id"].duplicated().sum()))
exigir(not df[["id", "host_id"]].isna().any().any(), "Hay identificadores de anuncio o anfitrión nulos")
# Solo se pueden retirar duplicados idénticos; no elegir una versión arbitraria.
exigir(not df.drop_duplicates().duplicated("id").any(), "Hay anuncios duplicados con atributos contradictorios")
df = df.drop_duplicates("id")

df["precio_texto_original"] = df["price"]  # se conserva el valor original
texto_precio = df["price"].astype("string").str.strip()
formato_valido = texto_precio.str.fullmatch(
    r"[+-]?\$?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?", na=False)
normalizado = (texto_precio.where(formato_valido)
               .str.replace("$", "", regex=False).str.replace(",", "", regex=False))
df["precio_cop"] = pd.to_numeric(normalizado, errors="coerce").astype(float)
df["precio_cop"] = df["precio_cop"].where(np.isfinite(df["precio_cop"]))
precio_ausente = texto_precio.isna() | texto_precio.eq("").fillna(False)
precio_malformado = ~precio_ausente & df["precio_cop"].isna()
registrar("calidad", "precio con formato no válido", int(precio_malformado.sum()))
registrar("calidad", "precio nulo (se conservan, marcados, fuera del análisis)", int(df["precio_cop"].isna().sum()))
registrar("calidad", "precio no positivo", int((df["precio_cop"] <= 0).sum()))

lp = np.log(df["precio_cop"].where(df["precio_cop"] > 0))
exigir(lp.notna().sum() >= 4, "Se necesitan al menos cuatro precios positivos para calcular cuartiles")
q1, q3 = lp.quantile([.25, .75])
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
df["precio_log"] = lp
df["outlier_precio"] = ((lp < lo) | (lp > hi)).fillna(False)
df["incluido_analisis"] = df["precio_cop"].gt(0) & ~df["outlier_precio"]
registrar("calidad", f"outliers de precio (COP {np.exp(lo):,.0f} a {np.exp(hi):,.0f})", int(df["outlier_precio"].sum()))
registrar("calidad", "anuncios incluidos en el análisis", int(df["incluido_analisis"].sum()))

[calidad] id duplicados: 0
[calidad] precio con formato no válido: 0
[calidad] precio nulo (se conservan, marcados, fuera del análisis): 711
[calidad] precio no positivo: 0
[calidad] outliers de precio (COP 36,352 a 693,268): 796
[calidad] anuncios incluidos en el análisis: 17998


## 3. Transformación de atributos

In [4]:
LOCALIDADES = {"Candelaria": "La Candelaria", "Ciudad Bolivar": "Ciudad Bolívar",
               "Engativa": "Engativá", "Fontibon": "Fontibón", "Los Martires": "Los Mártires",
               "San Cristobal": "San Cristóbal", "Usaquen": "Usaquén"}
TIPOS = {"Entire home/apt": "Vivienda completa", "Private room": "Habitación privada",
         "Shared room": "Habitación compartida", "Hotel room": "Habitación de hotel"}

df["localidad"] = df["neighbourhood_cleansed"].replace(LOCALIDADES)
df["tipo_alojamiento"] = df["room_type"].map(TIPOS)
registrar("transformacion", "tipos de alojamiento sin traducción", int(df["tipo_alojamiento"].isna().sum()))
exigir(not (df["localidad"].isna().any() or df["tipo_alojamiento"].isna().any()),
       "Hay localidades nulas o tipos de alojamiento sin clasificar")


def categoria(p):
    p = str(p).lower()
    if "hotel" in p or "hostel" in p or "bed and breakfast" in p:
        return "Hotel u hostal"
    if "loft" in p:
        return "Loft"
    if any(k in p for k in ("rental unit", "condo", "apartment")):
        return "Apartamento"
    if any(k in p for k in ("home", "house", "villa", "casa", "cabin", "townhouse")):
        return "Casa"
    return "Otro"


def banos(t):
    t = str(t).lower()
    m = re.search(r"(\d+(?:\.\d+)?)", t)
    if m:
        return float(m.group(1))
    return 0.5 if "half" in t else np.nan


def n_amen(s):
    try:
        return len(json.loads(s))
    except Exception:
        return np.nan


df["categoria_propiedad"] = df["property_type"].map(categoria)
df["banos"] = df["bathrooms_text"].map(banos)
df["banos_compartidos"] = df["bathrooms_text"].astype("string").str.lower().str.contains("shared")
df["n_amenidades"] = df["amenities"].map(n_amen)
df["tiene_licencia"] = df["license"].astype("string").str.strip().ne("").fillna(False)
df["superhost"] = df["host_is_superhost"].map({"t": "Sí", "f": "No"})
df["identidad_verificada"] = df["host_identity_verified"].map({"t": "Sí", "f": "No"})

[transformacion] tipos de alojamiento sin traducción: 0


## 4. Reseñas, métricas derivadas y segmentos
Sin reseñas ⇒ `resenas_por_mes = 0`; la calificación se mantiene **nula** (no se imputa).

In [5]:
for c in ["first_review", "last_review", "last_scraped"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["dias_desde_ultima_resena"] = (df["last_scraped"] - df["last_review"]).dt.days

df["tiene_resenas"] = df["number_of_reviews"] > 0
df["resenas_por_mes"] = df["reviews_per_month"].mask(
    df["reviews_per_month"].isna() & df["number_of_reviews"].eq(0), 0)
registrar("calidad", "reseñas por mes nulas con historial de reseñas (sin imputar)",
          int((df["reviews_per_month"].isna() & df["number_of_reviews"].gt(0)).sum()))
df["sin_actividad_12m"] = df["number_of_reviews_ltm"] == 0
registrar("transformacion", "anuncios sin calificación (se dejan nulos, no 0)",
          int(df["review_scores_rating"].isna().sum()))

df["precio_por_huesped"] = (df["precio_cop"].where(df["precio_cop"].gt(0))
                            / df["accommodates"].where(df["accommodates"].gt(0)))
registrar("calidad", "capacidad nula o no positiva (sin precio por huésped)",
          int((df["accommodates"].isna() | df["accommodates"].le(0)).sum()))
df["ocupacion_estimada_pct"] = df["estimated_occupancy_l365d"] / 365
registrar("transformacion", "máximo observado de ocupación estimada (días)",
          float(df["estimated_occupancy_l365d"].max()))

df["tipo_anfitrion"] = pd.cut(df["calculated_host_listings_count"], [0, 1, 4, np.inf],
                              labels=["Individual (1)", "Pequeño (2-4)", "Profesional (5+)"]).astype(str)
df["segmento_estadia"] = pd.cut(df["minimum_nights"], [0, 2, 6, 29, np.inf],
                                labels=["1-2 noches", "3-6 noches", "7-29 noches", "30+ noches"]).astype(str)
df["categoria_calificacion"] = pd.cut(
    df["review_scores_rating"], [0, 4.5, 4.8, 4.9, 5.01], right=False,
    labels=["Menor a 4.5", "4.5 a 4.79", "4.8 a 4.89", "4.9 o más"]).astype(str)
df.loc[df["review_scores_rating"].isna(), "categoria_calificacion"] = "Sin calificación"

cuartiles = df.loc[df["incluido_analisis"], "precio_cop"].quantile([.25, .5, .75]).values
df["segmento_precio"] = np.where(
    ~df["incluido_analisis"], "Fuera del análisis",
    np.select([df["precio_cop"] <= cuartiles[0], df["precio_cop"] <= cuartiles[1],
               df["precio_cop"] <= cuartiles[2]],
              ["1 Económico", "2 Medio-bajo", "3 Medio-alto"], "4 Premium"))
registrar("transformacion", "cortes de segmento de precio (COP, cuartiles)",
          " | ".join(f"{v:,.0f}" for v in cuartiles))

[calidad] reseñas por mes nulas con historial de reseñas (sin imputar): 0
[transformacion] anuncios sin calificación (se dejan nulos, no 0): 4089
[calidad] capacidad nula o no positiva (sin precio por huésped): 0
[transformacion] máximo observado de ocupación estimada (días): 255.0
[transformacion] cortes de segmento de precio (COP, cuartiles): 110,000 | 160,000 | 221,286


## 5. Modelo dimensional
Dimensiones (localidad, tipo de alojamiento, anfitrión) y tabla de hechos. Los atributos del anfitrión deben ser constantes por anfitrión.

In [6]:
dim_loc = (df.groupby("localidad")
           .agg(latitud_centro=("latitude", "mean"), longitud_centro=("longitude", "mean"),
                n_anuncios=("id", "size"), n_anuncios_analisis=("incluido_analisis", "sum"))
           .reset_index())
dim_loc["muestra_suficiente"] = dim_loc["n_anuncios_analisis"] >= MIN_N
dim_loc = dim_loc.sort_values("n_anuncios", ascending=False).reset_index(drop=True)
dim_loc.insert(0, "id_localidad", dim_loc.index + 1)

dim_tipo = (df[["tipo_alojamiento"]].drop_duplicates().sort_values("tipo_alojamiento")
            .reset_index(drop=True))
dim_tipo.insert(0, "id_tipo_alojamiento", dim_tipo.index + 1)

for c in ["superhost", "identidad_verificada", "hosts_time_as_host_years",
          "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
          "calculated_host_listings_count_private_rooms"]:
    inconsistentes = int((df.groupby("host_id")[c].nunique() > 1).sum())
    registrar("calidad", f"anfitriones con valores inconsistentes en {c}", inconsistentes)
    exigir(not inconsistentes, f"Resolver atributos contradictorios del anfitrión: {c}")
exigir(not (df["calculated_host_listings_count"].isna().any()
            or df["calculated_host_listings_count"].lt(1).any()),
       "La cantidad de anuncios por anfitrión debe ser positiva")

dim_host = (df.groupby("host_id").agg(
    superhost=("superhost", "first"), identidad_verificada=("identidad_verificada", "first"),
    anios_como_anfitrion=("hosts_time_as_host_years", "first"),
    n_anuncios=("calculated_host_listings_count", "max"),
    n_viviendas_completas=("calculated_host_listings_count_entire_homes", "max"),
    n_habitaciones_privadas=("calculated_host_listings_count_private_rooms", "max"),
    tipo_anfitrion=("tipo_anfitrion", "first")).reset_index()
    .rename(columns={"host_id": "id_host"}))

df = df.merge(dim_loc[["id_localidad", "localidad"]], on="localidad", how="left")
df = df.merge(dim_tipo, on="tipo_alojamiento", how="left")

RENOMBRAR = {
    "id": "id_anuncio", "host_id": "id_host", "latitude": "latitud", "longitude": "longitud",
    "property_type": "tipo_propiedad", "accommodates": "capacidad", "bedrooms": "habitaciones",
    "beds": "camas", "minimum_nights": "noches_minimas", "availability_365": "disponibilidad_365",
    "number_of_reviews": "resenas_total", "number_of_reviews_ltm": "resenas_12m",
    "number_of_reviews_l30d": "resenas_30d", "review_scores_rating": "calificacion",
    "review_scores_cleanliness": "calif_limpieza", "review_scores_location": "calif_ubicacion",
    "review_scores_value": "calif_valor", "first_review": "primera_resena",
    "last_review": "ultima_resena", "estimated_occupancy_l365d": "ocupacion_estimada_dias",
    "estimated_revenue_l365d": "ingreso_estimado_cop", "last_scraped": "fecha_captura"}
COLUMNAS_FACT = [
    "id_anuncio", "id_host", "id_localidad", "id_tipo_alojamiento", "categoria_propiedad",
    "tipo_propiedad", "latitud", "longitud", "capacidad", "habitaciones", "camas", "banos",
    "banos_compartidos", "n_amenidades", "noches_minimas", "segmento_estadia",
    "disponibilidad_365", "tiene_licencia", "precio_texto_original", "precio_cop", "precio_log",
    "precio_por_huesped", "segmento_precio", "outlier_precio", "incluido_analisis",
    "resenas_total", "resenas_12m", "resenas_30d", "resenas_por_mes", "tiene_resenas",
    "sin_actividad_12m", "calificacion", "categoria_calificacion", "calif_limpieza",
    "calif_ubicacion", "calif_valor", "primera_resena", "ultima_resena",
    "dias_desde_ultima_resena", "ocupacion_estimada_dias", "ocupacion_estimada_pct",
    "ingreso_estimado_cop", "fecha_captura"]
fact = df.rename(columns=RENOMBRAR)[COLUMNAS_FACT]

[calidad] anfitriones con valores inconsistentes en superhost: 0
[calidad] anfitriones con valores inconsistentes en identidad_verificada: 0
[calidad] anfitriones con valores inconsistentes en hosts_time_as_host_years: 0
[calidad] anfitriones con valores inconsistentes en calculated_host_listings_count: 0
[calidad] anfitriones con valores inconsistentes en calculated_host_listings_count_entire_homes: 0
[calidad] anfitriones con valores inconsistentes en calculated_host_listings_count_private_rooms: 0


## 6. Tablas derivadas
Tabla plana para Python, muestras de H1/H2, resumen por anfitrión (H5), excluidos y calidad de variables.

In [7]:
etiquetas = [(dim_loc[["id_localidad", "localidad"]], "id_localidad"),
             (dim_tipo, "id_tipo_alojamiento")]


def con_etiquetas(tabla):
    """Agrega los nombres de localidad y tipo a una tabla de hechos."""
    for dim, clave in etiquetas:
        tabla = tabla.merge(dim, on=clave)
    return tabla


plano = con_etiquetas(fact).merge(
    dim_host[["id_host", "superhost", "tipo_anfitrion", "n_anuncios"]], on="id_host")
inc = con_etiquetas(fact[fact["incluido_analisis"]])

muestra_h1 = filtrar_min_n(inc, "localidad")[
    ["id_anuncio", "id_host", "localidad", "precio_cop", "precio_log"]]
muestra_h2 = filtrar_min_n(inc, "tipo_alojamiento")[
    ["id_anuncio", "id_host", "tipo_alojamiento", "precio_cop", "precio_log"]]

anf_resumen = (inc.groupby("id_host").agg(
    n_anuncios_analizados=("id_anuncio", "size"),
    precio_mediano=("precio_cop", "median"),
    ocupacion_media_dias=("ocupacion_estimada_dias", "mean"),
    resenas_12m_media=("resenas_12m", "mean"),
    calificacion_mediana=("calificacion", "median")).reset_index()
    .merge(dim_host[["id_host", "n_anuncios", "tipo_anfitrion"]], on="id_host"))
anf_resumen["grupo_h5"] = np.where(anf_resumen["n_anuncios"] == 1, "1 anuncio", "2 o más anuncios")

excluidos = fact[~fact["incluido_analisis"]][
    ["id_anuncio", "id_host", "precio_texto_original", "precio_cop"]].copy()
texto_excluido = excluidos["precio_texto_original"].astype("string").str.strip()
excluidos["motivo_exclusion"] = np.select(
    [(texto_excluido.isna() | texto_excluido.eq("").fillna(False)).to_numpy(dtype=bool),
     excluidos["precio_cop"].isna(), excluidos["precio_cop"].le(0)],
    ["Precio nulo", "Precio con formato no válido", "Precio no positivo"],
    default="Precio atípico (1,5 x RIQ sobre log del precio)")

calidad = pd.DataFrame({
    "columna": raw.columns, "tipo_original": raw.dtypes.astype(str).values,
    "nulos": raw.isna().sum().values, "pct_nulos": (raw.isna().mean() * 100).round(2).values,
    "valores_distintos": raw.nunique().values})
calidad["conservada"] = calidad["columna"].isin(COLUMNAS)
calidad["completamente_vacia"] = calidad["nulos"] == len(raw)

informe = {
    "archivo_fuente": ENTRADA.name,
    "sha256_fuente": hashlib.sha256(ENTRADA.read_bytes()).hexdigest(),
    "fechas_captura": sorted(df["last_scraped"].dropna().dt.strftime("%Y-%m-%d").unique().tolist()),
    "registros_originales": int(len(raw)), "columnas_originales": int(raw.shape[1]),
    "columnas_vacias": int(len(vacias)), "columnas_conservadas": len(COLUMNAS),
    "anfitriones_distintos": int(raw["host_id"].nunique()),
    "precio_nulo": int(df["precio_cop"].isna().sum()),
    "precio_formato_invalido": int(precio_malformado.sum()),
    "precio_no_positivo": int(df["precio_cop"].le(0).sum()),
    "precio_atipico": int(df["outlier_precio"].sum()),
    "limite_inferior_cop": round(float(np.exp(lo)), 2),
    "limite_superior_cop": round(float(np.exp(hi)), 2),
    "anuncios_analisis": int(df["incluido_analisis"].sum()),
    "muestra_h1": int(len(muestra_h1)), "muestra_h2": int(len(muestra_h2)),
    "anfitriones_resumen": int(len(anf_resumen)),
    "anfitriones_1_anuncio": int((anf_resumen["grupo_h5"] == "1 anuncio").sum()),
    "anfitriones_2_o_mas": int((anf_resumen["grupo_h5"] != "1 anuncio").sum()),
    "calificaciones_nulas_total": int(df["review_scores_rating"].isna().sum()),
    "calificaciones_nulas_analisis": int(df.loc[df["incluido_analisis"], "review_scores_rating"].isna().sum()),
}

## 7. Validaciones y exportación

In [8]:
exigir(fact["id_anuncio"].is_unique, "Identificador del anuncio no único")
for dimension, clave in [(dim_loc, "id_localidad"), (dim_tipo, "id_tipo_alojamiento"), (dim_host, "id_host")]:
    exigir(dimension[clave].notna().all() and dimension[clave].is_unique, f"Clave de dimensión inválida: {clave}")
    exigir(fact[clave].isin(dimension[clave]).all(), f"Clave foránea inválida: {clave}")
exigir(len(plano) == len(fact), "El cruce multiplicó o perdió anuncios")
exigir(fact.loc[fact["incluido_analisis"], "precio_cop"].gt(0).all(), "La muestra analítica contiene precios no positivos")
exigir(len(inc) + len(excluidos) == len(fact), "Las muestras no concilian con el total")
exigir(np.isfinite(fact["precio_por_huesped"].dropna()).all(), "Hay divisiones no finitas en precio por huésped")
registrar("validacion", "integridad referencial y unicidad de id_anuncio", "OK")

TABLAS = {"fact_anuncios": fact, "dim_localidad": dim_loc, "dim_tipo_alojamiento": dim_tipo,
          "dim_anfitrion": dim_host, "airbnb_bogota_analitico": plano,
          "anfitriones_resumen": anf_resumen, "muestra_h1_localidad": muestra_h1,
          "muestra_h2_tipo_alojamiento": muestra_h2, "anuncios_excluidos": excluidos,
          "calidad_variables": calidad}
for nombre, tabla in TABLAS.items():
    guardar_csv(tabla, DIR_FASE3, nombre + ".csv")
    registrar("salida", f"{nombre}.csv (filas x columnas)", f"{tabla.shape[0]} x {tabla.shape[1]}")

registrar.guardar(DIR_FASE3)
(DIR_FASE3 / "informe_preparacion.json").write_text(
    json.dumps(informe, ensure_ascii=False, indent=2), encoding="utf-8")

[validacion] integridad referencial y unicidad de id_anuncio: OK
[salida] fact_anuncios.csv (filas x columnas): 19505 x 43
[salida] dim_localidad.csv (filas x columnas): 20 x 7
[salida] dim_tipo_alojamiento.csv (filas x columnas): 4 x 2
[salida] dim_anfitrion.csv (filas x columnas): 8497 x 8
[salida] airbnb_bogota_analitico.csv (filas x columnas): 19505 x 48
[salida] anfitriones_resumen.csv (filas x columnas): 7795 x 9
[salida] muestra_h1_localidad.csv (filas x columnas): 17945 x 5
[salida] muestra_h2_tipo_alojamiento.csv (filas x columnas): 17971 x 5
[salida] anuncios_excluidos.csv (filas x columnas): 1507 x 5
[salida] calidad_variables.csv (filas x columnas): 90 x 7


752

## 8. Diccionario de datos

In [9]:
D, M, A = "Dimensión", "Métrica", "Atributo"
diccionario = [
    ("fact_anuncios", "id_anuncio", "Clave", "Identificador del anuncio (id original de Inside Airbnb)"),
    ("fact_anuncios", "id_host", "Clave foránea", "Relaciona con dim_anfitrion"),
    ("fact_anuncios", "id_localidad", "Clave foránea", "Relaciona con dim_localidad"),
    ("fact_anuncios", "id_tipo_alojamiento", "Clave foránea", "Relaciona con dim_tipo_alojamiento"),
    ("fact_anuncios", "categoria_propiedad", D, "Agrupación de property_type: Apartamento, Casa, Loft, Hotel u hostal, Otro"),
    ("fact_anuncios", "tipo_propiedad", A, "property_type original (en inglés)"),
    ("fact_anuncios", "latitud / longitud", A, "Coordenadas del anuncio (para mapas)"),
    ("fact_anuncios", "capacidad", M, "accommodates: número de huéspedes"),
    ("fact_anuncios", "habitaciones / camas", M, "bedrooms y beds (nulos: camas 1,2 %)"),
    ("fact_anuncios", "banos", M, "Número de baños extraído de bathrooms_text (medio baño = 0,5)"),
    ("fact_anuncios", "banos_compartidos", A, "Verdadero si el texto indica 'shared'; nulo si no se informa bathrooms_text"),
    ("fact_anuncios", "n_amenidades", M, "Cantidad de amenidades listadas"),
    ("fact_anuncios", "noches_minimas", M, "minimum_nights"),
    ("fact_anuncios", "segmento_estadia", D, "1-2, 3-6, 7-29 o 30+ noches mínimas"),
    ("fact_anuncios", "disponibilidad_365", M, "Días disponibles en los próximos 365. No permite distinguir reservas de bloqueos del anfitrión"),
    ("fact_anuncios", "tiene_licencia", A, "Verdadero si license contiene texto no vacío. No acredita vigencia ni cumplimiento legal"),
    ("fact_anuncios", "precio_texto_original", A, "price tal como viene en el archivo (texto), conservado para auditoría"),
    ("fact_anuncios", "precio_cop", M, "price convertido a número, en COP. Cotización puntual, no histórico"),
    ("fact_anuncios", "precio_log", M, "Logaritmo natural de precio_cop (usado en el criterio de atípicos)"),
    ("fact_anuncios", "precio_por_huesped", M, "precio_cop / capacidad, solo cuando ambos son positivos; en otro caso queda nulo"),
    ("fact_anuncios", "segmento_precio", D, "Cuartiles del precio sobre anuncios incluidos: Económico, Medio-bajo, Medio-alto, Premium"),
    ("fact_anuncios", "outlier_precio", A, "Verdadero si el precio queda fuera de 1,5 x RIQ sobre log(precio)"),
    ("fact_anuncios", "incluido_analisis", A, "Verdadero si el precio es positivo, finito y no atípico. Filtro de la muestra común de precios; los conteos totales de oferta no lo requieren"),
    ("fact_anuncios", "resenas_total / resenas_12m / resenas_30d", M, "Reseñas acumuladas, de los últimos 12 meses y de los últimos 30 días"),
    ("fact_anuncios", "resenas_por_mes", M, "reviews_per_month: se reemplaza un nulo por 0 únicamente si number_of_reviews = 0"),
    ("fact_anuncios", "tiene_resenas / sin_actividad_12m", A, "Indicadores: alguna reseña histórica / cero reseñas en 12 meses"),
    ("fact_anuncios", "calificacion", M, "review_scores_rating (1-5). Nulo si no hay reseñas; NO se imputa"),
    ("fact_anuncios", "categoria_calificacion", D, "Sin calificación, <4.5, 4.5-4.79, 4.8-4.89, 4.9+"),
    ("fact_anuncios", "calif_limpieza / calif_ubicacion / calif_valor", M, "Subcalificaciones"),
    ("fact_anuncios", "primera_resena / ultima_resena", A, "Fechas de reseña"),
    ("fact_anuncios", "dias_desde_ultima_resena", M, "fecha_captura - ultima_resena"),
    ("fact_anuncios", "ocupacion_estimada_dias", M, "Estimación de Inside Airbnb derivada de reseñas. Máximo observado en esta captura: 255 días"),
    ("fact_anuncios", "ocupacion_estimada_pct", M, "ocupacion_estimada_dias / 365"),
    ("fact_anuncios", "ingreso_estimado_cop", M, "Estimación de Inside Airbnb en COP. No es ingreso real"),
    ("fact_anuncios", "fecha_captura", A, "last_scraped (2026-09-16)"),
    ("dim_localidad", "id_localidad, localidad", D, "20 localidades de Bogotá con nombres corregidos (tildes)"),
    ("dim_localidad", "latitud_centro / longitud_centro", A, "Promedio de coordenadas de sus anuncios"),
    ("dim_localidad", "n_anuncios, n_anuncios_analisis, muestra_suficiente", A, "Conteo total, conteo con precio incluido y marca de n_anuncios_analisis >= 30 para H1"),
    ("dim_tipo_alojamiento", "id_tipo_alojamiento, tipo_alojamiento", D, "Vivienda completa, Habitación privada, Habitación compartida, Habitación de hotel"),
    ("dim_anfitrion", "id_host", "Clave", "Identificador del anfitrión"),
    ("dim_anfitrion", "superhost / identidad_verificada", D, "Sí / No"),
    ("dim_anfitrion", "anios_como_anfitrion", M, "hosts_time_as_host_years (sustituye a host_since, que está vacío)"),
    ("dim_anfitrion", "n_anuncios", M, "calculated_host_listings_count"),
    ("dim_anfitrion", "n_viviendas_completas / n_habitaciones_privadas", M, "Anuncios del anfitrión por tipo"),
    ("dim_anfitrion", "tipo_anfitrion", D, "Individual (1), Pequeño (2-4), Profesional (5+)"),
    ("anfitriones_resumen", "id_host, grupo_h5", "Clave / Dimensión", "Un registro por anfitrión con al menos un anuncio incluido; grupo: 1 anuncio o 2 o más"),
    ("anfitriones_resumen", "precio_mediano, ocupacion_media_dias, resenas_12m_media, calificacion_mediana", M, "Indicadores por anfitrión usados en H5 (una observación por anfitrión)"),
    ("anfitriones_resumen", "n_anuncios, n_anuncios_analizados", M, "Anuncios reportados por el dataset y anuncios incluidos en el análisis"),
    ("muestra_h1_localidad", "todas", "Muestra", "Anuncios incluidos en localidades con n >= 30 (base de H1)"),
    ("muestra_h2_tipo_alojamiento", "todas", "Muestra", "Anuncios incluidos en tipos de alojamiento con n >= 30 (base de H2)"),
    ("anuncios_excluidos", "motivo_exclusion", A, "Precio nulo, con formato no válido, no positivo o atípico"),
    ("calidad_variables", "todas", "Control", "Tipo, nulos, % de nulos y valores distintos de las 90 columnas originales"),
]
guardar_csv(pd.DataFrame(diccionario, columns=["tabla", "columna", "rol", "descripcion"]),
            DIR_FASE3, "diccionario_datos.csv")

## 9. Resumen de control

In [10]:
print("Categoría de propiedad:\n", fact["categoria_propiedad"].value_counts().to_string())
print("\nTipo de anfitrión (anuncios):\n",
      fact.merge(dim_host[["id_host", "tipo_anfitrion"]], on="id_host")["tipo_anfitrion"].value_counts().to_string())
print("\nSegmento de precio:\n", fact["segmento_precio"].value_counts().to_string())
print("\nCalificación:\n", fact["categoria_calificacion"].value_counts().to_string())
print("\nLicencia:", round(fact["tiene_licencia"].mean() * 100, 1), "% con licencia")
print("Nulos en fact (%):\n", (fact.isna().mean() * 100).round(1)[lambda s: s > 0].to_string())

Categoría de propiedad:
 categoria_propiedad
Apartamento       14259
Casa               2541
Loft               1285
Hotel u hostal     1281
Otro                139

Tipo de anfitrión (anuncios):
 tipo_anfitrion
Profesional (5+)    9171
Individual (1)      6077
Pequeño (2-4)       4257

Segmento de precio:
 segmento_precio
2 Medio-bajo          4518
1 Económico           4501
4 Premium             4500
3 Medio-alto          4479
Fuera del análisis    1507

Calificación:
 categoria_calificacion
4.9 o más           5851
4.5 a 4.79          4404
Sin calificación    4089
4.8 a 4.89          2890
Menor a 4.5         2271

Licencia: 84.9 % con licencia
Nulos en fact (%):
 camas                        1.2
banos                        0.2
banos_compartidos            0.1
precio_texto_original        3.6
precio_cop                   3.6
precio_log                   3.6
precio_por_huesped           3.6
calificacion                21.0
calif_limpieza              21.0
calif_ubicacion             